# CelebA-HQ DDPM training

This notebook clones the repo from GitHub, downloads the Kaggle CelebA-HQ dataset to Google Drive, and trains with the same pipeline using the folder-based loader.

Recommended first run:
- keep `IMAGE_SIZE = 128`
- keep a fresh `RUN_NAME`
- make sure your Kaggle API token is available on Drive


## Mount Drive and clone repo

Update `REPO_URL` if needed.


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH = "main"
WORKDIR = Path("/content/Human-Faces-Generation-Diffusion-Models")

DRIVE_ROOT = Path("/content/drive/MyDrive/aml")
RUNS_ROOT = DRIVE_ROOT / "ddpm_runs"
RUN_NAME = "celebahq_run_001"
RUN_DIR = RUNS_ROOT / RUN_NAME

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

assert WORKDIR.exists(), f"Repo folder not found: {WORKDIR}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", WORKDIR)
print("Run dir:", RUN_DIR)


## Install dependencies


In [ ]:
!pip install -q datasets huggingface_hub tqdm matplotlib torchvision kaggle


## Configure Kaggle access

Place your `kaggle.json` on Drive first, for example in `MyDrive/aml/kaggle/kaggle.json`, then point `KAGGLE_JSON_ON_DRIVE` at it.


In [ ]:
KAGGLE_JSON_ON_DRIVE = DRIVE_ROOT / "kaggle" / "kaggle.json"

if not KAGGLE_JSON_ON_DRIVE.is_file():
    raise FileNotFoundError(f"Kaggle token not found: {KAGGLE_JSON_ON_DRIVE}")

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(KAGGLE_JSON_ON_DRIVE, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle token configured.")


## Download CelebA-HQ to Drive

This only needs to run again if you delete the extracted dataset folder.


In [ ]:
KAGGLE_DATASET = "badasstechie/celebahq-resized-256x256"
DATASET_ROOT = DRIVE_ROOT / "datasets"
DATASET_DIR = DATASET_ROOT / "celebahq_resized_256"

DATASET_ROOT.mkdir(parents=True, exist_ok=True)

if not DATASET_DIR.exists():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d "$KAGGLE_DATASET" -p "$DATASET_DIR" --unzip

image_count = sum(1 for p in DATASET_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print("Dataset directory:", DATASET_DIR)
print("Image files found:", image_count)
if image_count == 0:
    raise RuntimeError("No images found after download. Inspect DATASET_DIR contents.")


## Training config


In [ ]:
EPOCHS = 50
BATCH_SIZE = 16
IMAGE_SIZE = 128
LR = 1e-4
TIMESTEPS = 1000
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
SAMPLE_EVERY = 10
NUM_SAMPLE_IMAGES = 8
FOLDER_SUBSET_SIZE = 3000
FOLDER_TEST_SIZE = 300


## Run training


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)
if device != "cuda":
    raise RuntimeError("GPU is not active. In Colab, switch Runtime -> Change runtime type -> GPU.")

!python3 /content/Human-Faces-Generation-Diffusion-Models/src/training/train.py \
    --epochs $EPOCHS \
    --batch_size $BATCH_SIZE \
    --image_size $IMAGE_SIZE \
    --lr $LR \
    --timesteps $TIMESTEPS \
    --weight_decay $WEIGHT_DECAY \
    --grad_clip $GRAD_CLIP \
    --sample_every $SAMPLE_EVERY \
    --num_sample_images $NUM_SAMPLE_IMAGES \
    --dataset_source folder \
    --dataset_path "$DATASET_DIR" \
    --folder_subset_size $FOLDER_SUBSET_SIZE \
    --folder_test_size $FOLDER_TEST_SIZE \
    --device cuda \
    --save_dir "$RUN_DIR"


## Inspect saved outputs


In [ ]:
print("Checkpoint files:", sorted(str(p.name) for p in RUN_DIR.glob("*.pth")))
print("Sample grids:", sorted(str(p.name) for p in (RUN_DIR / "generated_samples").glob("*.png")))
print("Metadata file exists:", (RUN_DIR / "run_config.json").is_file())
print((RUN_DIR / "run_config.json").read_text())
